# 👂 KORA-lite — Notebook 3 : l'oreille fon

Beaucoup d'arnaques passent par **appel** ou par **vocal WhatsApp**, souvent en fon. Pour que KORA les analyse, il faut d'abord **transformer la voix en texte**. C'est la *reconnaissance vocale* (ASR, *Automatic Speech Recognition*).

**On utilise :** [`Professor/mms-300m-fongbe`](https://huggingface.co/Professor/mms-300m-fongbe), un modèle de Meta (MMS, 315 millions de paramètres) spécialisé pour le fon. Il écrit le fon **avec les tons** (`á`, `à`, `ǎ`) et les lettres `ɖ ɛ ɔ`.

**Ce que fait ce notebook :**
1. Il charge le modèle et le teste sur des enregistrements de **J'aime Ma Langue**, le corpus national béninois publié en novembre 2026, enregistré **sur téléphone** par des bénévoles.
2. Il mesure **notre propre taux d'erreur**, en conditions proches de la réalité (téléphone), au lieu de recopier celui de la fiche du modèle.
3. Il te permet de tester **tes propres enregistrements**.

**Pourquoi c'est un vrai résultat :** la fiche du modèle annonce 9,48 % d'erreurs sur les mots, mais mesurés sur ALFFA, des enregistrements propres. J'aime Ma Langue n'a **pas** servi à entraîner ce modèle : notre mesure est donc indépendante et plus réaliste.

⚠️ **Licence** : ce modèle est en **CC-BY-NC-4.0** (pas d'usage commercial). Parfait pour le challenge. Pour un produit vendu à un opérateur, il faudra entraîner notre propre oreille fon (voir la fin du notebook).

⚙️ GPU T4 conseillé. ⏱ ~10 minutes.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip -q install jiwer

In [ ]:
# ▶ Connexion à Google Drive et chemins du projet
# En Colab : le dossier KORA doit être dans "Mon Drive" (MyDrive/KORA).
# En local : le notebook utilise le dossier parent.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/KORA"
else:
    BASE = os.path.abspath("..")
sys.path.append(BASE)
for d in ["data", "modeles", "resultats"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("Dossier du projet :", BASE)
print("Contenu :", sorted(os.listdir(BASE)))
from kora_lib import enlever_tons

## 1. Charger le modèle de reconnaissance vocale

`pipeline` est un raccourci de la bibliothèque `transformers` : on lui donne un son, il rend du texte.

In [ ]:
import torch
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
asr_fon = pipeline("automatic-speech-recognition", model="Professor/mms-300m-fongbe",
                   device=DEVICE, chunk_length_s=20)
print("✅ Modèle chargé sur", "GPU" if DEVICE == 0 else "CPU (lent)")

## 2. Récupérer des enregistrements J'aime Ma Langue

Le corpus fait plus de 15 Go : on ne télécharge **pas tout**. On lit le début de la première archive **en streaming** et on s'arrête après `N_AUDIOS` fichiers. Ça prend environ une minute.

📚 *Source : J'aime Ma Langue — JML-fon-speech-v1 (CC-BY-SA-4.0), initiative nationale du Bénin (MTDI, ASIN, IIDiA). À citer dans la présentation.*

In [ ]:
import requests, tarfile, pandas as pd
from huggingface_hub import hf_hub_download

DEPOT = "JaimeMaLangue/JML-fon-speech-v1"
N_AUDIOS = 300
DOSSIER_AUDIO = "/content/jml" if IN_COLAB else "./jml"
os.makedirs(DOSSIER_AUDIO, exist_ok=True)

# Les transcriptions (le "corrigé" écrit par les linguistes)
transcriptions = pd.read_csv(hf_hub_download(DEPOT, "transcribed.csv", repo_type="dataset"))
print(f"{len(transcriptions):,} enregistrements transcrits dans le corpus")

# Lecture en streaming de la première archive
url = f"https://huggingface.co/datasets/{DEPOT}/resolve/main/audio_transcribed_000.tar"
recuperes = {}
with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    r.raw.decode_content = True
    with tarfile.open(fileobj=r.raw, mode="r|") as archive:
        for membre in archive:
            if membre.isfile() and membre.name.endswith(".wav"):
                chemin = os.path.join(DOSSIER_AUDIO, os.path.basename(membre.name))
                with open(chemin, "wb") as f:
                    f.write(archive.extractfile(membre).read())
                recuperes[membre.name] = chemin
                if len(recuperes) % 50 == 0:
                    print(f"  {len(recuperes)} fichiers", end="\r")
                if len(recuperes) >= N_AUDIOS:
                    break

echantillon = transcriptions[transcriptions["audio_path"].isin(recuperes)].copy()
echantillon["fichier"] = echantillon["audio_path"].map(recuperes)
print(f"\n✅ {len(echantillon)} enregistrements prêts · {echantillon['duration'].sum()/60:.1f} minutes · "
      f"{echantillon['speaker_id'].nunique()} voix différentes")
echantillon[["transcription", "speaker_id", "duration"]].head()

In [ ]:
# Écoute un exemple
from IPython.display import Audio, display
ex = echantillon.iloc[0]
print("Texte attendu :", ex["transcription"])
display(Audio(ex["fichier"]))

## 3. Transcrire et mesurer

On fait écrire au modèle ce qu'il entend, puis on compare avec le corrigé des linguistes.

**Les deux mesures :**
- **WER** (*Word Error Rate*, taux d'erreur sur les mots) : sur 100 mots, combien sont faux, oubliés ou en trop. **Plus c'est bas, mieux c'est.**
- **CER** (*Character Error Rate*) : la même chose, lettre par lettre.

On mesure **deux fois** :
- **avec les tons** : il faut aussi le bon ton (`á` ≠ `à`). C'est le plus exigeant.
- **sans les tons** : ce qui compte pour KORA, car on retire les tons avant de détecter les arnaques.

Avant de comparer, on nettoie les deux textes de la même façon : minuscules, ponctuation et guillemets retirés, même encodage Unicode.

In [ ]:
import re, time, unicodedata
import soundfile as sf
import jiwer

def nettoyer(texte, garder_tons=True):
    t = unicodedata.normalize("NFC", str(texte)).lower()
    t = re.sub(r"[\"“”«»,.;:!?()\[\]…\-–]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t if garder_tons else enlever_tons(t)

audios = [sf.read(f)[0] for f in echantillon["fichier"]]
debut = time.time()
sorties = asr_fon([{"raw": a, "sampling_rate": 16000} for a in audios], batch_size=8)
duree_calcul = time.time() - debut
echantillon["prediction"] = [s["text"] for s in sorties]

refs, hyps = echantillon["transcription"].tolist(), echantillon["prediction"].tolist()
mesures = {
    "n_enregistrements": len(echantillon),
    "minutes_audio": round(echantillon["duration"].sum() / 60, 1),
    "WER_avec_tons": jiwer.wer([nettoyer(t) for t in refs], [nettoyer(t) for t in hyps]),
    "CER_avec_tons": jiwer.cer([nettoyer(t) for t in refs], [nettoyer(t) for t in hyps]),
    "WER_sans_tons": jiwer.wer([nettoyer(t, False) for t in refs], [nettoyer(t, False) for t in hyps]),
    "CER_sans_tons": jiwer.cer([nettoyer(t, False) for t in refs], [nettoyer(t, False) for t in hyps]),
    "secondes_calcul_par_seconde_audio": duree_calcul / echantillon["duration"].sum(),
}
for k, v in mesures.items():
    print(f"{k:38s} {v:.1%}" if k.startswith(("WER", "CER")) else f"{k:38s} {v:.3f}" if isinstance(v, float) else f"{k:38s} {v}")

**Comment lire ces chiffres :**
- Si le WER sur J'aime Ma Langue est **plus haut** que les 9,48 % de la fiche, c'est normal et c'est un **résultat intéressant** : la voix au téléphone, avec des locuteurs variés, est plus difficile que des enregistrements de studio. C'est exactement ce qu'un jury sérieux veut entendre.
- `secondes_calcul_par_seconde_audio` < 1 signifie que KORA transcrit **plus vite que le temps réel** : un vocal de 30 s est transcrit en moins de 30 s.

In [ ]:
# Les meilleurs et les pires exemples
echantillon["wer"] = [jiwer.wer(nettoyer(r, False), nettoyer(h, False)) for r, h in zip(refs, hyps)]
pd.set_option("display.max_colwidth", 120)
print("✅ Les mieux transcrits")
display(echantillon.nsmallest(5, "wer")[["transcription", "prediction", "wer"]])
print("❌ Les plus difficiles")
display(echantillon.nlargest(5, "wer")[["transcription", "prediction", "wer", "duration"]])

In [ ]:
import json
echantillon[["id", "speaker_id", "duration", "transcription", "prediction", "wer"]].to_csv(
    f"{BASE}/resultats/oreille_fon_details.csv", index=False)
with open(f"{BASE}/resultats/oreille_fon_metriques.json", "w") as f:
    json.dump({"modele": "Professor/mms-300m-fongbe", "corpus": DEPOT, **mesures}, f, ensure_ascii=False, indent=2)
print("💾 resultats/oreille_fon_metriques.json")

## 4. Teste ta propre voix 🎙️

Enregistre un vocal (WhatsApp ou dictaphone), par exemple un faux appel d'arnaque en fon joué par l'équipe, puis envoie-le ici. Formats acceptés : wav, mp3, m4a, ogg.

💡 **Idée pour la démo** : enregistrez 20 à 30 faux appels d'arnaque en fon dans l'équipe. Ce sera votre **test en conditions réelles** et votre démo live.

In [ ]:
import librosa
if IN_COLAB:
    from google.colab import files
    envoyes = files.upload()
    for nom in envoyes:
        son, _ = librosa.load(nom, sr=16000, mono=True)     # le modèle attend du 16 kHz mono
        texte = asr_fon({"raw": son, "sampling_rate": 16000})["text"]
        print(f"🎧 {nom} ({len(son)/16000:.1f} s)\n📝 {texte}\n   sans tons : {enlever_tons(texte)}")

## 5. Pour aller plus loin (après le challenge) : notre propre oreille fon

Le modèle `mms-300m-fongbe` n'est pas utilisable commercialement (CC-BY-NC). La voie pour un produit :
1. Partir d'un modèle de base vocal sous licence libre (par exemple de la famille **wav2vec2 XLS-R** ; vérifier sa licence exacte).
2. L'entraîner sur **J'aime Ma Langue** (CC-BY-SA-4.0 : usage commercial autorisé, en citant la source). Il y a là des centaines d'heures de fon enregistrées au téléphone.
3. Ajouter des enregistrements d'appels réels (avec consentement) pour coller au cas d'usage.

🎤 **Au jury** : « Pour la démo, nous utilisons un modèle open source de recherche. Pour la version commerciale, nous entraînerons notre propre oreille fon sur la base vocale nationale J'aime Ma Langue. »

## ✅ Récapitulatif
- `resultats/oreille_fon_metriques.json` : **notre WER mesuré** sur J'aime Ma Langue (avec et sans tons)
- `resultats/oreille_fon_details.csv` : chaque phrase, avec le texte attendu et le texte transcrit

➡️ **Suite : Notebook 4 — la démo.**